Description of the method explored in this notebook:

One other goal could be that we have a feature space in which there are only unknown objects - for example objects from alerts of one night of observation, so the feature space would change every night - and we want to know which among these are more likely to be of the positive class.

To do this, we take known objects from the positive class as inputs to the model to see what are their nearest neighbors in the feature space. If an output appears multiple times (an object that would be the neighbor of multiple inputs), we could treat it as a higher level of confidence that it is of the positive class.  
Again, the efficiency of this model relies on how well separated are objects from different classes in the feature space.

To measure the accuracy of this model, we add known objects from the positive class (different from the inputs) in the feature space and check how often they get selected by the model.

---

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import normalize
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
import warnings
# Disabling FutureWarnings:
warnings.filterwarnings('ignore', category=FutureWarning)

# Feature names (without linear features):
feature_names = ['mean', 'weightedMean', 'std', 'median', 'amplitude', 'beyond1Std', 'cusum', 'IPR10',
                 'kurtosis', 'MPR40_5', 'MPR20_10', 'maxSlope', 'medianAbsDev', 'medianBRP10',
                 'percentAmplitude', 'meanVariance', 'andersonDarlingNorm', 'chi2', 'skew', 'stetsonK']

In [2]:
# The following paths should be changed accordingly. The data can be saved in a csv from the clean_data.ipynb notebook.
positive = pd.read_csv('../../data/one_year_data/positive_g.csv', index_col=0)
negative = pd.read_csv('../../data/one_year_data/negative_g.csv', index_col=0)

# Adding labels for each class:
positive['class'] = 'positive'
negative['class'] = 'negative'

"""
# Normalizing features:
g_set = pd.concat([positive, negative]) # Grouping positive & negative together so that they are then normalized the same way.
g_set = normalize(g_set[feature_names], axis=0)
positive[feature_names] = g_set[:len(positive)]
negative[feature_names] = g_set[len(positive):]
"""

'\n# Normalizing features:\ng_set = pd.concat([positive, negative]) # Grouping positive & negative together so that they are then normalized the same way.\ng_set = normalize(g_set[feature_names], axis=0)\npositive[feature_names] = g_set[:len(positive)]\nnegative[feature_names] = g_set[len(positive):]\n'

In [3]:
# positive train will be in the feature space with negative objects
# positive test will be the input to the nearest neighbors algorithm
positive_train, positive_test = train_test_split(positive, train_size=.7, random_state=42)

In [5]:
t = pd.concat([negative, positive_train]).reset_index(drop=True)
t

,objectId,time_range (yr),nb_of_points,mean,weightedMean,std,median,amplitude,beyond1Std,cusum,...,maxSlope,medianAbsDev,medianBRP10,percentAmplitude,meanVariance,andersonDarlingNorm,chi2,skew,stetsonK,class
0,ZTF18acvqiot,5.2,48,16.803662,16.732781,0.244653,16.750022,0.663873,0.208333,0.179795,...,0.351817,0.147663,0.229167,0.901372,0.014560,0.762713,11.943453,1.094601,0.772226,negative
1,ZTF18acpdrdt,5.3,107,16.247058,16.145497,0.280431,16.222143,0.673854,0.280374,0.143019,...,8.728864,0.172428,0.186916,0.902676,0.017260,1.270051,32.241160,0.881849,0.712618,negative
2,ZTF22abtsipc,2.1,4,19.052173,18.978403,0.275456,19.011926,0.332430,0.500000,0.338237,...,0.008988,0.129969,0.500000,0.412924,0.014458,0.132593,4.660389,0.850724,0.765788,negative
3,ZTF18aachppe,4.5,23,20.028906,20.022994,0.159417,19.982714,0.308625,0.260870,0.212201,...,0.189886,0.109339,0.086957,0.379718,0.007959,0.520390,0.963497,0.426946,0.828548,negative
4,ZTF18abqrjvo,4.3,69,19.014286,18.996443,0.196618,19.006744,0.418896,0.304348,0.245331,...,103.998422,0.118639,0.202899,0.464953,0.010341,0.422395,1.776484,0.328756,0.766614,negative
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
412,ZTF17aaajocf,5.3,252,17.627373,17.563588,0.246879,17.615593,1.268181,0.087302,0.059228,...,1.862945,0.015192,0.896825,1.295160,0.014005,55.249149,18.358361,1.073920,0.440843,positive
413,ZTF18acrwtsm,5.4,256,16.900904,16.749162,0.450322,16.895900,1.214525,0.300781,0.288754,...,11.284382,0.236759,0.269531,1.423860,0.026645,1.428934,126.605097,0.309292,0.791544,positive
414,ZTF17aacbihx,5.1,15,17.928888,17.703110,0.384181,18.005972,0.802480,0.266667,0.166527,...,0.193967,0.205131,0.200000,0.912235,0.021428,0.228777,28.890926,-0.242072,0.763825,positive
415,ZTF18aaadlpa,5.2,131,18.183411,16.935745,1.406246,18.131498,2.142755,0.503817,0.085002,...,178.464798,1.359047,0.091603,2.206941,0.077337,4.258831,270.832648,0.005683,0.901037,positive


In [7]:
neigh = NearestNeighbors(n_neighbors=1).fit(t[feature_names])
neighbors_distance, neighbors_indices = neigh.kneighbors(positive_test[feature_names])

neighbors = t.loc[neighbors_indices.flatten()]

In [22]:
ids, counts = np.unique(neighbors['objectId'], return_counts=True)

col = ['objectId', 'time_range (yr)', 'nb_of_points', 'class']
candidates = pd.DataFrame(columns=[*col, 'count'])

for id, count in zip(ids, counts):
    if count > 1:
        candidates = pd.concat([candidates, neighbors[neighbors['objectId'] == id][col].iloc[0:1]])
        candidates['count'].iloc[-1] = count

/tmp/ipykernel_103960/3846448894.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  candidates['count'].iloc[-1] = count
/tmp/ipykernel_103960/3846448894.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  candidates['count'].iloc[-1] = count
/tmp/ipykernel_103960/3846448894.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  candidates['count'].iloc[-1] = count


In [ ]:
candidates

,objectId,time_range (yr),nb_of_points,class,count
375,ZTF18aabpzjg,5.0,226,positive,3
353,ZTF18abvlguz,5.3,28,negative,2
403,ZTF20abragvw,0.3,47,positive,2
